# ML - REGRESION LINEAL REGULARIZADA

In [1]:
import numpy as np
import pandas as pd

## Paso 1: Definicion del problema (target)
Decidimos que la variable obejtivo es `Heart disease_prevalence` ya que está relacionada con la salud y con varias variables predictoras

## Paso 2: Recopilacion de datos
[fuente](https://raw.githubusercontent.com/4GeeksAcademy/regularized-linear-regression-project-tutorial/main/demographic_health_data.csv)

### Importamos los datos y creamos el DataFrame

In [2]:
df = pd.read_csv('../data/raw/demographic_health_data.csv', index_col='COUNTY_NAME')
df[['Heart disease_prevalence','Heart disease_number']]

,Heart disease_prevalence,Heart disease_number
COUNTY_NAME,,
Autauga,7.9,3345
Baldwin,7.8,13414
Barbour,11.0,2159
Bibb,8.6,1533
Blount,9.2,4101
...,...,...
Sweetwater,5.9,1862
Teton,5.2,981
Uinta,7.2,1034


fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,40-49,40-49 y/o % of total pop,50-59,50-59 y/o % of total pop,60-69,60-69 y/o % of total pop,70-79,70-79 y/o % of total pop,80+,80+ y/o % of total pop,White-alone pop,% White-alone,Black-alone pop,% Black-alone,Native American/American Indian-alone pop,% NA/AI-alone,Asian-alone pop,% Asian-alone,Hawaiian/Pacific Islander-alone pop,% Hawaiian/PI-alone,Two or more races pop,% Two or more races,POP_ESTIMATE_2018,N_POP_CHG_2018,GQ_ESTIMATES_2018,R_birth_2018,R_death_2018,R_NATURAL_INC_2018,R_INTERNATIONAL_MIG_2018,R_DOMESTIC_MIG_2018,R_NET_MIG_2018,Less than a high school diploma 2014-18,High school diploma only 2014-18,Some college or associate's degree 2014-18,Bachelor's degree or higher 2014-18,Percent of adults with less than a high school diploma 2014-18,Percent of adults with a high school diploma only 2014-18,Percent of adults completing some college or associate's degree 2014-18,Percent of adults with a bachelor's degree or higher 2014-18,POVALL_2018,PCTPOVALL_2018,PCTPOV017_2018,PCTPOV517_2018,MEDHHINC_2018,CI90LBINC_2018,CI90UBINC_2018,Civilian_labor_force_2018,Employed_2018,Unemployed_2018,Unemployment_rate_2018,Median_Household_Income_2018,Med_HH_Income_Percent_of_State_Total_2018,Active Physicians per 100000 Population 2018 (AAMC),Total Active Patient Care Physicians per 100000 Population 2018 (AAMC),Active Primary Care Physicians per 100000 Population 2018 (AAMC),Active Patient Care Primary Care Physicians per 100000 Population 2018 (AAMC),Active General Surgeons per 100000 Population 2018 (AAMC),Active Patient Care General Surgeons per 100000 Population 2018 (AAMC),Total nurse practitioners (2019),Total physician assistants (2019),Total Hospitals (2019),Internal Medicine Primary Care (2019),Family Medicine/General Practice Primary Care (2019),Total Specialist Physicians (2019),ICU Beds_x,Total Population,Population Aged 60+,Percent of Population Aged 60+,COUNTY_NAME,STATE_NAME,STATE_FIPS,CNTY_FIPS,county_pop2018_18 and older,anycondition_prevalence,anycondition_Lower 95% CI,anycondition_Upper 95% CI,anycondition_number,Obesity_prevalence,Obesity_Lower 95% CI,Obesity_Upper 95% CI,Obesity_number,Heart disease_prevalence,Heart disease_Lower 95% CI,Heart disease_Upper 95% CI,Heart disease_number,COPD_prevalence,COPD_Lower 95% CI,COPD_Upper 95% CI,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code

In [3]:
# Importamos los datos de la información de las variables
df_info_columns = pd.read_csv('/workspaces/pabloaznar-intro-ml/data/raw/data_dict.csv', index_col='Column')
df_info_columns
#df_info_columns.loc['flips']

,description,info
Column,,
fips,FIPS Code for the County,NaN
TOT_POP,Total Population,This data as well as all Age and Race data is ...
0-9,Population aged 0-9,All of the other age columns are the same but ...
0-9 y/o % of total pop,% of the population aged 0-9,NaN
10-19',NaN,NaN
...,...,...
NaN,"7 (Urban population of 2,500 to 19,999, not ad...",NaN
NaN,"8 (Completely rural or less than 2,500 urban p...",NaN
NaN,"9 (Completely rural or less than 2,500 urban p...",NaN


## Paso 3: Realizamos un EDA completo

### Analisis descriptivo

In [4]:
# Obtenemos las dimensiones
df.shape

(3140, 107)

In [ ]:
# Obtenemos informacion sobre el tipo de datos y valores no nulos
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3140 entries, Autauga to Weston
Columns: 107 entries, fips to Urban_rural_code
dtypes: float64(61), int64(45), object(1)
memory usage: 2.6+ MB


In [5]:
# Obtenemos información estadística de las variables numércias
df.describe().T

,count,mean,std,min,25%,50%,75%,max
fips,3140.0,30401.640764,15150.559265,1001.0,18180.500000,29178.000000,45081.50000,5.604500e+04
TOT_POP,3140.0,104189.412420,333583.395432,88.0,10963.250000,25800.500000,67913.00000,1.010552e+07
0-9,3140.0,12740.302866,41807.301846,0.0,1280.500000,3057.000000,8097.00000,1.208253e+06
0-9 y/o % of total pop,3140.0,11.871051,2.124081,0.0,10.594639,11.802727,12.95184,2.546068e+01
19-Oct,3140.0,13367.976752,42284.392134,0.0,1374.500000,3274.000000,8822.25000,1.239139e+06
...,...,...,...,...,...,...,...,...
CKD_prevalence,3140.0,3.446242,0.568059,1.8,3.100000,3.400000,3.80000,6.200000e+00
CKD_Lower 95% CI,3140.0,3.207516,0.527740,1.7,2.900000,3.200000,3.50000,5.800000e+00
CKD_Upper 95% CI,3140.0,3.710478,0.613069,1.9,3.300000,3.700000,4.10000,6.600000e+00
CKD_number,3140.0,2466.234076,7730.422067,3.0,314.750000,718.000000,1776.25000,2.377660e+05


In [7]:
# Obtenemos información estadística del target
df['Heart disease_prevalence'].describe()

count    3140.000000
mean        8.607803
std         1.758587
min         3.500000
25%         7.400000
50%         8.600000
75%         9.800000
max        15.100000
Name: Heart disease_prevalence, dtype: float64

> ## Observaciones:
>
> - Existen un total de 3140 filas (en este caso, condados) y 108 columnas
> - Los datos constan de 106 variables numéricas y 2 categóricas

### Limpieza de datos

In [6]:
# Comprobamos si hay valores duplicados
df[df.duplicated()]

,fips,TOT_POP,0-9,0-9 y/o % of total pop,19-Oct,10-19 y/o % of total pop,20-29,20-29 y/o % of total pop,30-39,30-39 y/o % of total pop,...,COPD_number,diabetes_prevalence,diabetes_Lower 95% CI,diabetes_Upper 95% CI,diabetes_number,CKD_prevalence,CKD_Lower 95% CI,CKD_Upper 95% CI,CKD_number,Urban_rural_code
COUNTY_NAME,,,,,,,,,,,,,,,,,,,,,


In [7]:
# Filtros para verificar las columnas que vamos a borrar
cols_a_borrar_pos = list(range(2, 32, 2))
cols_a_borrar = [df.columns[i] for i in cols_a_borrar_pos]

print("Columnas a borrar:", cols_a_borrar)

Columnas a borrar: ['0-9', '19-Oct', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80+', 'White-alone pop', 'Black-alone pop', 'Native American/American Indian-alone pop', 'Asian-alone pop', 'Hawaiian/Pacific Islander-alone pop', 'Two or more races pop']


In [14]:
df.columns.get_loc('Less than a high school diploma 2014-18')

26

In [8]:
# Elimanos las variables numericas con los totales/conteos y nos quedamos con la misma informacion pero en porcentajes para evitar redundancia
# De la variable que esta en la posicion 2 a la 32 los totales se encuentran en las posiciones impares
df = df.drop(columns=cols_a_borrar)
#df.drop(['0-9', '19-Oct', '20-29','30-39', '40-49', '50-59', '60-69', '70-79', '80+', 'White-alone pop', 'Black-alone pop',  ], axis=1, inplace=True)
df.columns

Index(['fips', 'TOT_POP', '0-9 y/o % of total pop', '10-19 y/o % of total pop',
       '20-29 y/o % of total pop', '30-39 y/o % of total pop',
       '40-49 y/o % of total pop', '50-59 y/o % of total pop',
       '60-69 y/o % of total pop', '70-79 y/o % of total pop',
       '80+ y/o % of total pop', '% White-alone', '% Black-alone',
       '% NA/AI-alone', '% Asian-alone', '% Hawaiian/PI-alone',
       '% Two or more races', 'POP_ESTIMATE_2018', 'N_POP_CHG_2018',
       'GQ_ESTIMATES_2018', 'R_birth_2018', 'R_death_2018',
       'R_NATURAL_INC_2018', 'R_INTERNATIONAL_MIG_2018', 'R_DOMESTIC_MIG_2018',
       'R_NET_MIG_2018', 'Less than a high school diploma 2014-18',
       'High school diploma only 2014-18',
       'Some college or associate's degree 2014-18',
       'Bachelor's degree or higher 2014-18',
       'Percent of adults with less than a high school diploma 2014-18',
       'Percent of adults with a high school diploma only 2014-18',
       'Percent of adults completing som

In [9]:
cols_a_borrar_pos_2 = list(range(26, 30))
cols_a_borrar_2 = [df.columns[i] for i in cols_a_borrar_pos_2]

print("Columnas a borrar:", cols_a_borrar_2)

Columnas a borrar: ['Less than a high school diploma 2014-18', 'High school diploma only 2014-18', "Some college or associate's degree 2014-18", "Bachelor's degree or higher 2014-18"]


In [ ]:
df = df.drop(columns=cols_a_borrar_2)


KeyError: '[\'Less than a high school diploma 2014-18\', \'High school diploma only 2014-18\', "Some college or associate\'s degree 2014-18", "Bachelor\'s degree or higher 2014-18"] not found in axis'

In [15]:
df.shape

(3140, 88)

> ## Conclusiones:
>
> - No existen registros duplicados
> - Eliminamos 19 columnas de variables totales o conteos para evitar redundancia ya que las tenemos en porcentajes
> - Ahora pasamos a tener 88 variables predictoras